# 03 — Baseline Models

Run LightGBM, XGBoost, and CatBoost with 5-fold GroupKFold CV. External data injected into every training fold. Target: AUC > 0.945 with LGB before tuning.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import roc_auc_score

from src.features import get_feature_cols, TARGET
from src.cv import run_cv, add_target_encoding
from src.models import make_lgb_fn, make_xgb_fn, make_cat_fn
from src.utils import save_submission

plt.rcParams['figure.dpi'] = 120

# Load cached features (run notebook 02 first)
cache_dir = Path('..') / 'cache'
train_feat = pd.read_pickle(cache_dir / 'train_feat.pkl')
test_feat  = pd.read_pickle(cache_dir / 'test_feat.pkl')
ext_feat   = pd.read_pickle(cache_dir / 'ext_feat.pkl')

print(f'Loaded: train={train_feat.shape}, test={test_feat.shape}, ext={ext_feat.shape}')

In [ ]:
# Add target encoding (out-of-fold, no leakage)
print('Computing out-of-fold target encodings...')
train_feat, ext_feat, test_feat = add_target_encoding(train_feat, ext_feat, test_feat)

feature_cols = get_feature_cols(train_feat)
# Add target encoding columns
te_cols = [c for c in ['Driver_TE', 'Race_TE', 'Race_Year_TE'] if c in train_feat.columns]
feature_cols = feature_cols + te_cols
print(f'Total features: {len(feature_cols)}')

## 1. LightGBM Baseline

In [ ]:
print('=== LightGBM 5-fold GroupKFold CV ===')
lgb_fn = make_lgb_fn()
lgb_oof, lgb_fold_aucs = run_cv(train_feat, ext_feat, feature_cols, lgb_fn)
print(f'\nLGB OOF AUC: {roc_auc_score(train_feat[TARGET], lgb_oof):.5f}')

## 2. XGBoost Baseline

In [ ]:
print('=== XGBoost 5-fold GroupKFold CV ===')
xgb_fn = make_xgb_fn()
xgb_oof, xgb_fold_aucs = run_cv(train_feat, ext_feat, feature_cols, xgb_fn)
print(f'\nXGB OOF AUC: {roc_auc_score(train_feat[TARGET], xgb_oof):.5f}')

## 3. CatBoost Baseline

In [ ]:
print('=== CatBoost 5-fold GroupKFold CV ===')
cat_fn = make_cat_fn()
cat_oof, cat_fold_aucs = run_cv(train_feat, ext_feat, feature_cols, cat_fn)
print(f'\nCAT OOF AUC: {roc_auc_score(train_feat[TARGET], cat_oof):.5f}')

## 4. Simple rank-average ensemble

In [ ]:
from src.utils import rank_avg

ensemble_oof = rank_avg([lgb_oof, xgb_oof, cat_oof])
print(f'Rank ensemble OOF AUC: {roc_auc_score(train_feat[TARGET], ensemble_oof):.5f}')

# Summary table
results = pd.DataFrame({
    'Model': ['LGB', 'XGB', 'CatBoost', 'Rank Ensemble'],
    'OOF_AUC': [
        roc_auc_score(train_feat[TARGET], lgb_oof),
        roc_auc_score(train_feat[TARGET], xgb_oof),
        roc_auc_score(train_feat[TARGET], cat_oof),
        roc_auc_score(train_feat[TARGET], ensemble_oof),
    ]
}).sort_values('OOF_AUC', ascending=False)
print(results.to_string(index=False))

## 5. Feature importance (LightGBM)

In [ ]:
import lightgbm as lgb_lib

# Train one final LGB on full train+ext to get importances
X_all = pd.concat([train_feat[feature_cols], ext_feat[feature_cols]])
y_all = pd.concat([train_feat[TARGET], ext_feat[TARGET]])

from src.models import DEFAULT_LGB_PARAMS
imp_model = lgb_lib.LGBMClassifier(**{**DEFAULT_LGB_PARAMS, 'n_estimators': 500})
imp_model.fit(X_all, y_all)

fi = pd.Series(imp_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, max(6, len(feature_cols) * 0.3)))
fi.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('LightGBM feature importance (gain)')
plt.tight_layout()
plt.show()

## 6. Train full models and generate test predictions

In [ ]:
from src.models import train_full
import numpy as np

X_all_arr = X_all.values
y_all_arr = y_all.values
w_all_arr = np.concatenate([np.ones(len(train_feat)), np.full(len(ext_feat), 0.8)])

X_test = test_feat[feature_cols].values

print('Training full LGB for test predictions...')
lgb_full = train_full(X_all_arr, y_all_arr, w_all_arr, model_name='lgb', n_estimators=500)
lgb_test_preds = lgb_full.predict_proba(X_test)[:, 1]

print('Training full XGB for test predictions...')
xgb_full = train_full(X_all_arr, y_all_arr, w_all_arr, model_name='xgb', n_estimators=500)
xgb_test_preds = xgb_full.predict_proba(X_test)[:, 1]

print('Training full CatBoost for test predictions...')
cat_full = train_full(X_all_arr, y_all_arr, w_all_arr, model_name='cat', n_estimators=500)
cat_test_preds = cat_full.predict_proba(X_test)[:, 1]

ensemble_test = rank_avg([lgb_test_preds, xgb_test_preds, cat_test_preds])
print('Test predictions generated.')

In [ ]:
# Save OOF and test predictions for ensemble notebook
import pickle

oof_preds = {'lgb': lgb_oof, 'xgb': xgb_oof, 'cat': cat_oof}
test_preds = {'lgb': lgb_test_preds, 'xgb': xgb_test_preds, 'cat': cat_test_preds}

with open(cache_dir / 'oof_preds_baseline.pkl', 'wb') as f:
    pickle.dump(oof_preds, f)
with open(cache_dir / 'test_preds_baseline.pkl', 'wb') as f:
    pickle.dump(test_preds, f)

# Save baseline submission
sub_path = save_submission(test_feat['id'], ensemble_test, tag='baseline_rank_ensemble')
print(f'Baseline submission saved: {sub_path}')